# Sprint 4 — Graph A GCN Runner

**Runner-only notebook. Hiçbir model, preprocessing, evaluation veya plot logic içermez.**
Tüm bilimsel kod `src/`, `scripts/`, `configs/` altındadır.

Slice 4B execution plan: `docs/exec-plans/active/004-sprint4-gcn-baseline.md`  
Runner boundary: `colab/README.md`

---
**Başlamadan önce kontrol et:**
- [ ] Colab runtime: GPU seçildi mi? (Çalışma Zamanı → Çalışma Zamanı Türünü Değiştir → T4 GPU)
- [ ] Drive'da `crispr_gnn_offtarget/data/processed/graphs/sprint3/` klasörü var mı?
- [ ] Branch GitHub'a push edildi mi?

## ADIM 1 — Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ADIM 2 — Repo Clone ve Checkout

In [ ]:
%%bash
pip install uv --quiet
git clone https://github.com/YasinEkici/crispr-gnn-offtarget.git crispr-gnn-offtarget
cd crispr-gnn-offtarget
git checkout sprint4/gcn-baseline
echo "=== Commit SHA ==="
git rev-parse HEAD

## ADIM 3 — Dependency Sync ve Sürüm Kontrolü

⚠️ `cuda_available: False` çıkarsa GPU runtime seçilmemiş veya uv CPU torch kurmuş demektir. Training yine çalışır ama yavaş olur ve sonuç provisional sayılır.

In [ ]:
%%bash
cd crispr-gnn-offtarget
uv sync
echo "=== Sürüm Kontrolü ==="
uv run python -c "
import torch, torch_geometric
print('torch         :', torch.__version__)
print('pyg           :', torch_geometric.__version__)
print('cuda_available:', torch.cuda.is_available())
print('cuda_version  :', torch.version.cuda)
print('device        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
"

## ADIM 4 — Run ID ve Config Güncelle

Tarihi güncelle. Aynı gün ikinci run yaparsan `_v2` ekle.

In [ ]:
RUN_ID = "sprint4_graph_a_gcn_seed42_20260530"  # ← tarihi güncelle
print("Run ID:", RUN_ID)

In [ ]:
from pathlib import Path

import torch
import yaml

base_config_path = Path("crispr-gnn-offtarget/configs/experiments/gcn_minimal.yaml")
run_dir = Path(f"crispr-gnn-offtarget/outputs/sprint4/graph_a/runs/{RUN_ID}")
run_dir.mkdir(parents=True, exist_ok=True)
resolved_config_path = run_dir / "resolved_config.yaml"

with open(base_config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
config["run_id"] = RUN_ID
config.setdefault("training", {})["device"] = device

with open(resolved_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print(f"Base config unchanged: {base_config_path}")
print(f"Resolved run config:  {resolved_config_path}")
print(f"Device: {device}")
if device == "cpu":
    print("⚠️  CUDA yok — training CPU'da çalışacak, sonuç provisional sayılır.")

print("\nResolved fields:")
print(f"run_id: {config['run_id']}")
print(f"device: {config['training']['device']}")
print(f"scheduler: {config['training'].get('scheduler')}")
print(f"clip_grad_norm: {config['training'].get('clip_grad_norm')}")

## ADIM 5 — Sprint 3 Artifact'lerini Drive'dan Kopyala

Drive'da `crispr_gnn_offtarget/data/processed/graphs/sprint3/` klasörü olmalı.

In [ ]:
%%bash
DRIVE_SPRINT3="/content/drive/MyDrive/crispr_gnn_offtarget/data/processed/graphs/sprint3"
LOCAL_GRAPHS="crispr-gnn-offtarget/data/processed/graphs"

mkdir -p "${LOCAL_GRAPHS}"
cp -r "${DRIVE_SPRINT3}" "${LOCAL_GRAPHS}/"

echo "=== Kopyalanan klasörler ==="
ls "${LOCAL_GRAPHS}/sprint3/"

## ADIM 6 — Provenance Gate (Zorunlu)

⛔ Bu adım hata verirse ADIM 7'yi başlatma.

In [ ]:
import subprocess, json, os

provenance_path = f"crispr-gnn-offtarget/outputs/sprint4/graph_a/runs/{RUN_ID}/graph_artifact_provenance.json"
os.makedirs(f"crispr-gnn-offtarget/outputs/sprint4/graph_a/runs/{RUN_ID}", exist_ok=True)

result = subprocess.run(
    [
        "uv", "run", "python", "scripts/validate_graph_artifacts.py",
        "--artifact-dir", "data/processed/graphs/sprint3",
        "--approved-source", "drive_sprint3_handoff",
        "--output", f"outputs/sprint4/graph_a/runs/{RUN_ID}/graph_artifact_provenance.json",
    ],
    cwd="crispr-gnn-offtarget",
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("HATA:", result.stderr)
    raise RuntimeError("Provenance gate başarısız — ADIM 7'yi başlatma!")

with open(provenance_path) as f:
    prov = json.load(f)
print("\n=== Schema Özeti ===")
for schema, info in prov["schemas"].items():
    print(f"{schema[:35]:35} | edges: {info['candidate_pair_edges']} | split: {info['split_id']}")
print("\n✅ Provenance gate geçti — training başlatılabilir.")

## ADIM 7 — Graph A Training

Training bitince aşağıdakiler otomatik yazılır:
- `outputs/sprint4/graph_a/runs/<run_id>/resolved_config.yaml`
- `outputs/sprint4/graph_a/runs/<run_id>/runtime.json`
- `outputs/sprint4/graph_a/runs/<run_id>/model.pt`
- `outputs/sprint4/graph_a/runs/<run_id>/training_history.csv`
- `outputs/sprint4/graph_a/gcn_graph_a_results.csv`
- `outputs/sprint4/graph_a/gcn_graph_a_report.md`
- `outputs/sprint4/graph_a/diagnostics/`
- `outputs/sprint4/graph_a/figures/`

⚠️ Training bittikten sonra test metriklerine bakabilirsin ama hiçbir parametre, epoch veya threshold değiştirme.

In [ ]:
import subprocess

result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "scripts/train.py",
        "--config",
        f"outputs/sprint4/graph_a/runs/{RUN_ID}/resolved_config.yaml",
    ],
    cwd="crispr-gnn-offtarget",
    text=True,
)
if result.returncode != 0:
    raise RuntimeError("Graph A training command failed")

## ADIM 8 — Artifact'leri Drive'a Kopyala

In [ ]:
import shutil, os

DRIVE_OUT = f"/content/drive/MyDrive/crispr_gnn_offtarget/returned_outputs/{RUN_ID}"
os.makedirs(DRIVE_OUT, exist_ok=True)

base = "crispr-gnn-offtarget"

copies = [
    (f"{base}/outputs/sprint4/graph_a/runs/{RUN_ID}", f"{DRIVE_OUT}/{RUN_ID}"),
    (f"{base}/outputs/sprint4/graph_a/gcn_graph_a_results.csv", f"{DRIVE_OUT}/gcn_graph_a_results.csv"),
    (f"{base}/outputs/sprint4/graph_a/gcn_graph_a_report.md",   f"{DRIVE_OUT}/gcn_graph_a_report.md"),
    (f"{base}/outputs/sprint4/graph_a/diagnostics",             f"{DRIVE_OUT}/diagnostics_sprint4_graph_a"),
    (f"{base}/outputs/sprint4/graph_a/figures",                 f"{DRIVE_OUT}/figures_sprint4_graph_a"),
]

for src, dst in copies:
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif os.path.isfile(src):
        shutil.copy2(src, dst)
    else:
        print(f"⚠️  Bulunamadı: {src}")
        continue
    print(f"✅ {src} → {dst}")

## ADIM 9 — Kopyalanan Artifact Kontrolü

In [ ]:
import os

DRIVE_OUT = f"/content/drive/MyDrive/crispr_gnn_offtarget/returned_outputs/{RUN_ID}"

required = [
    f"{DRIVE_OUT}/{RUN_ID}/graph_artifact_provenance.json",
    f"{DRIVE_OUT}/{RUN_ID}/resolved_config.yaml",
    f"{DRIVE_OUT}/{RUN_ID}/runtime.json",
    f"{DRIVE_OUT}/{RUN_ID}/model.pt",
    f"{DRIVE_OUT}/{RUN_ID}/training_history.csv",
    f"{DRIVE_OUT}/gcn_graph_a_results.csv",
    f"{DRIVE_OUT}/gcn_graph_a_report.md",
]

required_figures = [
    "gcn_graph_a_graph_schema_auprc_comparison.png",
    "gcn_graph_a_pr_curves.png",
    "gcn_graph_a_roc_curves.png",
    "gcn_graph_a_training_curves.png",
    "gcn_graph_a_score_distributions.png",
    "gcn_graph_a_confusion_matrices.png",
    "gcn_graph_a_decile_lift.png",
    "gcn_graph_a_per_genome_metrics.png",
    "gcn_graph_a_view_sanity_example.png",
]

required_diagnostics = [
    "gcn_graph_a_predictions.csv",
    "gcn_graph_a_training_history.csv",
    "gcn_graph_a_score_direction.csv",
    "gcn_graph_a_fixed_threshold_metrics.csv",
    "gcn_graph_a_score_deciles.csv",
]

all_ok = True
print("=== Zorunlu Artifact'ler ===")
for path in required:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {os.path.basename(path)}")
    if not exists:
        all_ok = False

print("\n=== Zorunlu Figürler (figures_sprint4_graph_a/) ===")
for fig in required_figures:
    path = f"{DRIVE_OUT}/figures_sprint4_graph_a/{fig}"
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {fig}")
    if not exists:
        all_ok = False

print("\n=== Zorunlu Diagnostic Tablolar (diagnostics_sprint4_graph_a/) ===")
for diag in required_diagnostics:
    path = f"{DRIVE_OUT}/diagnostics_sprint4_graph_a/{diag}"
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {diag}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("✅ Tüm artifact'ler mevcut — Slice 4C için hazır.")
else:
    print("❌ Eksik artifact'ler var — yukarıdaki listeyi kontrol et.")